# Checkpoint — Anomaliedetectie & data drift

Dit notebook wordt automatisch gecheckt via pytest (GitHub Actions). De checks voeren je notebook uit en valideren of bepaalde variabelen bestaan en binnen de verwachte ranges vallen. **Je krijgt geen hints in de foutmeldingen: het is aan jou om correct werkende code te schrijven.**

## BELANGRIJK: gebruik exact de genoemde variabelennamen

De checks zoeken volgende variabelen in je notebook. Andere namen worden niet gezien:

- `data` — de netwerkdataset als pandas DataFrame (`network.csv`)
- `y_pred_anomaly` — de IsolationForest-voorspellingen (rij `1` = normaal, `-1` = anomalie)
- `n_anomalies_if` — het aantal gedetecteerde anomalieën (aantal `-1`-labels)
- `p_value_feature` — de p-waarde van de Kolmogorov-Smirnov-test
- `data_drift_detected` — boolean (`True`/`False`) of er feature drift is
- `DriftDetector` — de klasse voor ML-driftdetectie (met methoden `fit` en `detect_drift`)
- `drift_detected_vb` — het voorbeeldresultaat van de driftdetector
- `antwoord_drift` — meerkeuzevraag: `"A"`, `"B"`, `"C"` of `"D"`

Gebruik voor het anomaliemodel `IsolationForest(contamination=0.07, random_state=42)`. De data ligt in dezelfde map als dit notebook, dus `pd.read_csv('network.csv')` volstaat.

## Oefening 1 — Anomaliedetectie op netwerkdata (Isolation Forest)

We laden de netwerkdataset `network.csv` met kolommen `Duration`, `SrcBytes` en `DstBytes`. We trainen een **Isolation Forest** (`contamination=0.07, random_state=42`) op deze kenmerken en gebruiken het model om per observatie te voorspellen of ze normaal is (`1`) of een anomalie (`-1`).

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# Dataset inladen
data = pd.read_csv('network.csv')

# Kenmerken voor het model
X = data[['Duration', 'SrcBytes', 'DstBytes']]

In [ ]:
# TODO: maak een Isolation Forest-model met contamination=0.07 en random_state=42,
# train het op X en bewaar de voorspellingen in y_pred_anomaly.
# Bereken daarna het aantal -1-labels en stop dat in n_anomalies_if.

model = IsolationForest(contamination=0.07, random_state=42)
model.fit(X)

y_pred_anomaly = model.predict(X)
n_anomalies_if = 0

for value in y_pred_anomaly:
    if value == -1:
        n_anomalies_if += 1

print(f"Het aantal outliers is: {n_anomalies_if}")


Het aantal outliers is: 7


## Oefening 2 — Feature drift opsporen met een KS-test

We vertrekken van een synthetische tijdreeksen-dataset met drie features over het jaar 2020. Het **huidige venster** is de laatste 7 dagen aan data; de rest is **historisch**. Gebruik een **Kolmogorov-Smirnov-test** (`scipy.stats.ks_2samp`) om na te gaan of de verdeling van `Feature_A` significant verschilt. Een p-waarde lager dan `0.05` betekent feature drift.

In [8]:
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

# dagelijkse data over 2020
np.random.seed(42)
date_range = pd.date_range(start='2020-01-01', end='2020-12-31', freq='D')

# Random data
df = pd.DataFrame({
    'Date': date_range,
    'Feature_A': np.random.randn(len(date_range)),
    'Feature_B': np.random.randn(len(date_range)),
    'Feature_C': np.random.randn(len(date_range)),
})

In [9]:
# Bepaal het cutoff-moment: de laatste 7 dagen zijn het 'huidige venster',
# de rest is 'historisch'. Doe dit voor Feature_A.
cutoff = df['Date'].max() - pd.Timedelta(days=7)
historical_feature = df.loc[df['Date'] < cutoff, 'Feature_A'].values
current_feature = df.loc[df['Date'] >= cutoff, 'Feature_A'].values

/tmp/ipykernel_4499/1582151844.py:3: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  cutoff = df['Date'].max() - pd.Timedelta(days=7)


In [16]:
# TODO: voer de KS-test uit op Feature_A (historisch vs. huidig).
# Bewaar de p-waarde in p_value_feature.
# Zet data_drift_detected op True als de p-waarde < 0.05 is, anders False.

test  = ks_2samp(historical_feature, current_feature)

p_value_feature = test.pvalue
data_drift_detected = p_value_feature < 0.05

print(p_value_feature)
print(data_drift_detected)

0.582709150052983
False


## Oefening 3 — ML-driftdetectie met de klasse `DriftDetector`

We bouwen een klasse `DriftDetector` die automatisch drift kan detecteren. De klasse krijgt een `significance_level` (standaard `0.05`) en slaat bij `fit(reference_data)` historische referentiedata op. `detect_drift(new_data)` voert een **KS-test** uit tussen de referentie en de nieuwe data en geeft `True` terug bij een p-waarde kleiner dan het significance level, anders `False`.

In [17]:
import numpy as np
from scipy.stats import ks_2samp


class DriftDetector:
    def __init__(self, significance_level=0.05):
        self.significance_level = significance_level
        self.reference_data = None
   
    def fit(self, reference_data):
        # TODO: sla de referentiedata op in self.reference_data
        self.reference_data = reference_data
   
    def detect_drift(self, new_data):
        # TODO: voer de KS-test uit tussen self.reference_data en new_data,
        # geef True terug als de p-waarde < self.significance_level is.
        return ks_2samp(self.reference_data, new_data).pvalue < self.significance_level


Hieronder testen we de detector op een duidelijke verschuiving (loc=3.0 t.o.v. loc=0):

In [18]:
np.random.seed(0)
reference_data = np.random.normal(loc=0, scale=1, size=5000)
verplaatst = np.random.normal(loc=3.0, scale=1, size=5000)

detector = DriftDetector(significance_level=0.05)
detector.fit(reference_data)

# TODO: roep detector.detect_drift(verplaatst) aan en bewaar het resultaat
# (True/False) in drift_detected_vb.
drift_detected_vb = detector.detect_drift(verplaatst)
print(drift_detected_vb)


True


## Meerkeuzevraag

Bij **feature drift** verandert de verdeling van de inputs `X` terwijl `p(y|X)` gelijk blijft. Bij **concept drift** verandert de relatie tussen input en output (`p(y|X)` wijzigt). Welke uitspraak is **juist**?

- **A)** Feature drift kun je detecteren met een KS-test op de inputfeatures.
- **B)** Concept drift laat de verdeling van de inputs ongewijzigd.
- **C)** Een KS-test kan concept drift direct aantonen.
- **D)** Feature drift verandert altijd de decision boundary.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_drift`.

In [ ]:
# antwoord_drift = "A", "B", "C" of "D"
antwoord_drift = "A"
